In [ ]:
# Store Sales Time Series Forecasting - Favorita Ecuador
# Kaggle Competition Solution

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# For modeling
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

In [ ]:
# Load all datasets
train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv', parse_dates=['date'])
test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv', parse_dates=['date'])
stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv', parse_dates=['date'])
holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv', parse_dates=['date'])
transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv', parse_dates=['date'])


print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Stores shape: {stores.shape}")
print(f"Oil shape: {oil.shape}")
print(f"Holidays shape: {holidays.shape}")

In [ ]:
# Display basic info
print("\nTrain data sample:")
print(train.head())

print("\nTrain data info:")
print(train.info())

print("\nBasic statistics:")
print(train.describe())

print("\nStores data:")
print(stores.head())

print("\nDate range in train:", train['date'].min(), "to", train['date'].max())
print("Date range in test:", test['date'].min(), "to", test['date'].max())

# Check for missing values
print("\nMissing values in train:")
print(train.isnull().sum())

print("\nUnique values:")
print(f"Stores: {train['store_nbr'].nunique()}")
print(f"Product families: {train['family'].nunique()}")

In [ ]:
# Handle oil prices - forward fill missing values
oil['dcoilwtico'] = oil['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')

# Create a complete date range and merge oil prices
date_range = pd.DataFrame({'date': pd.date_range(start=oil['date'].min(), 
                                                   end=oil['date'].max())})
oil_complete = date_range.merge(oil, on='date', how='left')
oil_complete['dcoilwtico'] = oil_complete['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')

# Process holidays
# Focus on national holidays and create a simple holiday indicator
holidays_national = holidays[holidays['locale'] == 'National'].copy()
holidays_national['is_holiday'] = 1

In [ ]:
print("\n" + "="*80)
print("FEATURE ENGINEERING")
print("="*80)

def create_features(df, is_train=True):
    """Create time-based and other features"""
    df = df.copy()
    
    # Date features
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    
    # Cyclical features
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['day_sin'] = np.sin(2 * np.pi * df['day'] / 31)
    df['day_cos'] = np.cos(2 * np.pi * df['day'] / 31)
    
    # Weekend indicator
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    
    # Month start/end indicators
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    
    # Payroll days (15th and end of month)
    df['is_payroll_day'] = ((df['day'] == 15) | (df['is_month_end'] == 1)).astype(int)
    
    # Days before payroll
    df['days_to_payroll'] = df['day'].apply(lambda x: min(15 - x if x <= 15 else 31 - x, 
                                                            abs(15 - x), abs(31 - x)))
    
    # Earthquake feature (April 16, 2016 onwards)
    earthquake_date = pd.Timestamp('2016-04-16')
    df['days_since_earthquake'] = (df['date'] - earthquake_date).dt.days
    df['post_earthquake'] = (df['days_since_earthquake'] >= 0).astype(int)
    df['earthquake_impact'] = np.where(
        (df['days_since_earthquake'] >= 0) & (df['days_since_earthquake'] <= 60),
        1, 0
    )
    
    # Merge with stores
    df = df.merge(stores, on='store_nbr', how='left')
    
    # Merge with oil prices
    df = df.merge(oil_complete, on='date', how='left')
    df['dcoilwtico'] = df['dcoilwtico'].fillna(method='ffill').fillna(method='bfill')
    
    # Merge with holidays
    df = df.merge(holidays_national[['date', 'is_holiday']], on='date', how='left')
    df['is_holiday'] = df['is_holiday'].fillna(0).astype(int)
    
    # Promotion features
    df['onpromotion'] = df['onpromotion'].fillna(0)
    
    return df

# Apply feature engineering
print("Creating features for train data...")
train_featured = create_features(train, is_train=True)

print("Creating features for test data...")
test_featured = create_features(test, is_train=False)

print(f"\nTrain featured shape: {train_featured.shape}")
print(f"Test featured shape: {test_featured.shape}")

In [ ]:
print("\n" + "="*80)
print("PREPARING DATA FOR MODELING")
print("="*80)

# Encode categorical variables
label_encoders = {}
categorical_cols = ['family', 'city', 'state', 'type']

for col in categorical_cols:
    le = LabelEncoder()
    train_featured[col] = le.fit_transform(train_featured[col].astype(str))
    test_featured[col] = le.transform(test_featured[col].astype(str))
    label_encoders[col] = le

# Select features for modeling
feature_cols = ['store_nbr', 'family', 'onpromotion', 
                'year', 'month', 'day', 'dayofweek', 'dayofyear', 
                'weekofyear', 'quarter', 'is_weekend', 'is_month_start', 
                'is_month_end', 'is_payroll_day', 'days_to_payroll',
                'month_sin', 'month_cos', 'day_sin', 'day_cos',
                'city', 'state', 'type', 'cluster',
                'dcoilwtico', 'is_holiday', 'post_earthquake', 
                'earthquake_impact', 'days_since_earthquake']

X = train_featured[feature_cols]
y = train_featured['sales']

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data - use last 15 days as validation
val_date = train_featured['date'].max() - timedelta(days=14)
train_mask = train_featured['date'] < val_date
val_mask = train_featured['date'] >= val_date

X_train = X[train_mask]
y_train = y[train_mask]
X_val = X[val_mask]
y_val = y[val_mask]

print(f"\nTrain set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")

In [ ]:
print("\n" + "="*80)
print("MODEL TRAINING")
print("="*80)

# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

# Parameters
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1
}

print("Training LightGBM model...")
model = lgb.train(
    params,
    train_data,
    num_boost_round=1000,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(100)]
)

print(f"\nBest iteration: {model.best_iteration}")
print(f"Best score: {model.best_score}")

In [ ]:

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(feature_importance.head(20))

In [ ]:
# Predict on validation set
y_val_pred = model.predict(X_val, num_iteration=model.best_iteration)
y_val_pred = np.maximum(y_val_pred, 0)  # Ensure non-negative predictions

# Calculate RMSE
val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
print(f"Validation RMSE: {val_rmse:.4f}")

# Calculate RMSLE (if no zeros in actual values)
y_val_nonzero = y_val > 0
if y_val_nonzero.sum() > 0:
    val_rmsle = np.sqrt(mean_squared_log_error(
        y_val[y_val_nonzero], 
        y_val_pred[y_val_nonzero]
    ))
    print(f"Validation RMSLE: {val_rmsle:.4f}")

In [ ]:
X_test = test_featured[feature_cols]
test_predictions = model.predict(X_test, num_iteration=model.best_iteration)
test_predictions = np.maximum(test_predictions, 0)  # Ensure non-negative

print(f"Test predictions shape: {test_predictions.shape}")
print(f"Prediction range: [{test_predictions.min():.2f}, {test_predictions.max():.2f}]")

In [ ]:
# Prepare submission
submission = test[['id']]
submission['sales'] = test_predictions

# Save submission
submission.to_csv('submission.csv', index=False)
print("Submission file saved as 'submission.csv'")
print("\nSubmission preview:")
print(submission.head(10))